# Chapter 10 -- 第三个分类器：CNN

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chaosfrey-arch/news-classification-tutorial/blob/main/chapter10_cnn.ipynb)

**本章目标：** 理解 1D 卷积在文本上的作用，搭建并训练 TextCNN，对比前面模型的结果。

**预计时间：** 60 分钟

> 原始论文：[Kim 2014 -- Convolutional Neural Networks for Sentence Classification](https://arxiv.org/abs/1408.5882)

## 10.1 为什么用 CNN 处理文本？

CNN（卷积神经网络）最初是为图像设计的，但在文本上也很有效。

**直觉：** 文本分类中，关键信息往往是**局部短语**，而不是全文的整体结构：
- "scored a goal" → Sports
- "stock market crash" → Business
- "launched a rocket" → Science/Tech

**1D 卷积核** 就像一个滑动窗口，扫描文章里的每一段短语（n-gram），检测是否存在某种模式。

类比：用放大镜扫描文章，每次看 3-5 个词，判断这段话是否包含某个主题信号。

## 10.2 1D 卷积的工作原理

```
文章（padding 后）：[w1, w2, w3, w4, w5, ..., w100]
每个 wi 是 128 维向量（来自 Embedding 层）

卷积核大小 = 3（即每次看 3 个词）：
  位置 1-3：[w1, w2, w3] → 一个分数（这 3 个词的「激活强度」）
  位置 2-4：[w2, w3, w4] → 一个分数
  ...
  位置 98-100：[w98, w99, w100] → 一个分数

结果：100-3+1 = 98 个分数，组成一个特征向量
```

**GlobalMaxPooling1D**：从 98 个分数里取最大值 → 1 个数
- 含义：这个卷积核「检测的模式」在文章中**出现过的最强信号**
- 不管模式出现在文章哪个位置，都能被捕获（位置不变性）

## 10.3 多尺度卷积（Multi-scale Filters）

Kim 2014 论文的核心设计：**同时用多种大小的卷积核**

```
Embedding 层输出 (100, 128)
    ↓              ↓              ↓
Conv1D(128, 3)  Conv1D(128, 4)  Conv1D(128, 5)   ← 同时检测 3、4、5 个词的短语
    ↓              ↓              ↓
GlobalMaxPool  GlobalMaxPool  GlobalMaxPool
    ↓              ↓              ↓
 [128 维]       [128 维]       [128 维]
    └──────────────────────────────┘
              拼接（Concatenate）
                  [384 维]
                     ↓
              Dense(4, softmax)
                  [4 个类别]
```

每种大小的卷积核捕获不同粒度的语言模式，合并后信息更丰富。

In [ ]:
import pandas as pd, numpy as np, re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

def clean_text(t):
    t = str(t).lower()
    t = re.sub(r'http\S+|www\S+|https\S+', '', t)
    t = re.sub(r'[^a-z\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

df = pd.read_csv('dataset.csv').dropna(subset=['Class']).copy()
df['Description'] = df['Description'].fillna('')
df['text'] = (df['Title'] + ' ' + df['Description']).apply(clean_text)

le = LabelEncoder()
y = le.fit_transform(df['Class'])
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['text'].values, y, test_size=0.3, random_state=42, stratify=y)

VOCAB_SIZE = 20000
MAX_LEN    = 100
EMBED_DIM  = 128

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_text)

X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train_text),
                             maxlen=MAX_LEN, padding='pre', truncating='pre')
X_test_pad  = pad_sequences(tokenizer.texts_to_sequences(X_test_text),
                             maxlen=MAX_LEN, padding='pre', truncating='pre')

print('数据准备完毕！')
print(f'X_train_pad: {X_train_pad.shape}, X_test_pad: {X_test_pad.shape}')

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Embedding, Conv1D, GlobalMaxPooling1D,
                                      Dense, Dropout, Concatenate)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

y_train_cat = to_categorical(y_train, 4)
y_test_cat  = to_categorical(y_test,  4)

# 多尺度 TextCNN（Kim 2014 风格）
NUM_FILTERS = 128  # 每种卷积核的数量
FILTER_SIZES = [3, 4, 5]  # 三种尺度

inputs = Input(shape=(MAX_LEN,), name='input')
emb = Embedding(VOCAB_SIZE, EMBED_DIM, name='embedding')(inputs)

# 三路卷积并行
branches = []
for fs in FILTER_SIZES:
    x = Conv1D(NUM_FILTERS, fs, activation='relu', name=f'conv_{fs}')(emb)
    x = GlobalMaxPooling1D(name=f'pool_{fs}')(x)
    branches.append(x)

merged = Concatenate(name='concat')(branches)     # 拼接三路特征
x = Dropout(0.5, name='dropout')(merged)
outputs = Dense(4, activation='softmax', name='output')(x)

cnn_model = Model(inputs, outputs, name='TextCNN')
cnn_model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
cnn_model.summary()

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = cnn_model.fit(
    X_train_pad, y_train_cat,
    epochs=15,
    batch_size=256,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)
print(f'实际训练了 {len(history.history["loss"])} 轮')

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import seaborn as sns

# 训练曲线
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, (metric, title) in enumerate([('accuracy', 'Accuracy'), ('loss', 'Loss')]):
    axes[i].plot(history.history[metric], label='Train')
    axes[i].plot(history.history[f'val_{metric}'], label='Validation')
    axes[i].set_title(f'TextCNN -- {title}', fontsize=13)
    axes[i].set_xlabel('Epoch'); axes[i].set_ylabel(title)
    axes[i].legend(); axes[i].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 评估
y_pred_prob = cnn_model.predict(X_test_pad, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)

acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred, average='macro')
print(f'TextCNN  Accuracy: {acc:.4f}  |  Macro F1: {f1:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=le.classes_))

# 混淆矩阵
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('TextCNN -- Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# 保存模型和 tokenizer 供 Chapter 12 使用
import joblib
cnn_model.save('cnn_model.keras')
joblib.dump(tokenizer, 'tokenizer.pkl')
print('已保存：cnn_model.keras, tokenizer.pkl')

## 总结

| 概念 | 含义 |
|------|------|
| Conv1D(n, k) | n 个卷积核，每个看 k 个词的局部窗口 |
| GlobalMaxPooling1D | 取序列上的最大值，输出定长向量 |
| 多尺度卷积 | 同时用 3、4、5 个词的窗口，捕获不同粒度短语 |
| Concatenate | 拼接多路特征 |
| Dropout(0.5) | 训练时随机关闭 50% 的神经元，防止过拟合 |

**TextCNN 预期结果：Accuracy ≈ 91-92%**（比 TF-IDF + NB 略强）

**下一章 →** Chapter 11：第四个分类器——双向 LSTM（BiLSTM）